In [11]:
import numpy as np
import numpy.testing as npt

from scipy import stats
from scipy.stats import t,ttest_ind
from scipy.stats import f
from scipy.stats import f_oneway
from scipy.stats import pearsonr

import statsmodels.api as sm
from statsmodels.regression._prediction import get_prediction
from statsmodels.stats.outliers_influence import OLSInfluence,MLEInfluence
from statsmodels.graphics.gofplots import qqplot_2samples,ProbPlot,qqplot
import pandas as pd
from patsy import dmatrices
from numpy.testing import assert_almost_equal, assert_allclose
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import seaborn as sns
import os
# some_file.py
import sys
# caution: path[0] is reserved for script path (or '' in REPL)
sys.path.insert(1, r'C:\Users\TODO\Desktop\Abhi\AI\AI\Math\Hands-On\statemodelsStudy')
from olsRegressionAnalysis import dispAnalysisOfVariance, tableDispFormatt,getInvOfProductMat,\
                                  getRegressionEqn,\
                                  dispReghressionAnalysis,norm_scalling,getCorrelation,\
                                      get_variance_inflation_factors

In [12]:
path = os.path.join(os.getcwd(), 'DataSet', 'DeliveryTimeData.txt')
df = pd.read_csv(path)
print(df.columns)

Index(['Obs', 'DlvrTime_Y', 'NumCase_X1', 'Dist_X2'], dtype='object')


In [13]:
formulas = 'DlvrTime_Y ~ NumCase_X1 + Dist_X2'

dfParams = pd.DataFrame({})
dfWeight = pd.DataFrame({})

y, X = dmatrices(
                 formula_like = formulas, 
                 data=df,
                 return_type='dataframe'
                 )

res = sm.OLS(y, X).fit()
tableDispFormatt('ols estimator')
print(res.params)

=============================== ols estimator ==============================================
Intercept     2.341231
NumCase_X1    1.615907
Dist_X2       0.014385
dtype: float64


In [14]:
dfParams['OLS'] = res.params
#dfWeight['OLS'] = res.weights

huber_t = sm.RLM(y,X, M=sm.robust.norms.HuberT(t= 2.0))
hub_results = huber_t.fit()
tableDispFormatt('Robust Regressions Criterion = HuberT')
print(hub_results.params)
#print(hub_results.chisq)
#print(hub_results.weights)

=============================== Robust Regressions Criterion = HuberT ======================
Intercept     3.366267
NumCase_X1    1.509258
Dist_X2       0.013917
dtype: float64


In [15]:
dfParams['HUBER_T'] = hub_results.params
dfWeight['Obs'] = hub_results.weights.index.values + 1
dfWeight['HUBER_T'] = hub_results.weights


RamsayEa = sm.RLM(y,X, M=sm.robust.norms.RamsayE(a = 0.3))
RamsayEa_result = RamsayEa.fit()
tableDispFormatt('Robust Regressions Criterion = RamsayEa')
print(RamsayEa_result.params)

=============================== Robust Regressions Criterion = RamsayEa ====================
Intercept     3.906808
NumCase_X1    1.472154
Dist_X2       0.012840
dtype: float64


In [16]:
dfParams['RamsayE'] = RamsayEa_result.params
dfWeight['RamsayE'] = RamsayEa_result.weights

HampelFunction = sm.RLM(y,X, M=sm.robust.norms.Hampel(a=1.7, b=3.4, c=8.5))
HampelFunction_result = HampelFunction.fit()
tableDispFormatt('Robust Regressions Criterion = RamsayEa')
print(HampelFunction_result.params)

=============================== Robust Regressions Criterion = RamsayEa ====================
Intercept     4.328056
NumCase_X1    1.482490
Dist_X2       0.011022
dtype: float64


In [17]:
dfParams['Hampel'] = HampelFunction_result.params
dfWeight['Hampel'] = HampelFunction_result.weights

andrew_wave = sm.RLM(y,X, M=sm.robust.norms.AndrewWave(a = 1.48))
andrew_wave_result = andrew_wave.fit()
tableDispFormatt('Robust Regressions Criterion = AndrewWave')
print(andrew_wave_result.params)
#print(andrew_wave_result.chisq)
#print(andrew_wave_result.weights)

=============================== Robust Regressions Criterion = AndrewWave ==================
Intercept     4.467879
NumCase_X1    1.479986
Dist_X2       0.010612
dtype: float64


In [18]:
dfParams['AndrewWave'] = andrew_wave_result.params
dfWeight['AndrewWave'] = andrew_wave_result.weights

tukeyBiweight = sm.RLM(y,X, M=sm.robust.norms.TukeyBiweight())
tukeyBiweight_result = tukeyBiweight.fit()
tableDispFormatt('Robust Regressions Criterion = Tukey\'s Biweight')
print(tukeyBiweight_result.params)
#print(tukeyBiweight_result.chisq)
#print(tukeyBiweight_result.weights)

=============================== Robust Regressions Criterion = Tukey's Biweight ============
Intercept     4.470980
NumCase_X1    1.475521
Dist_X2       0.010693
dtype: float64


In [19]:
dfParams['Trukey'] = tukeyBiweight_result.params
dfWeight['Trukey'] = tukeyBiweight_result.weights


dfParams.index = ['Const','Case','Dist']

tableDispFormatt('PARAMS')
print(dfParams)
tableDispFormatt('WEIGHT')
print(dfWeight)

=============================== PARAMS =====================================================
            OLS   HUBER_T   RamsayE    Hampel  AndrewWave    Trukey
Const  2.341231  3.366267  3.906808  4.328056    4.467879  4.470980
Case   1.615907  1.509258  1.472154  1.482490    1.479986  1.475521
Dist   0.014385  0.013917  0.012840  0.011022    0.010612  0.010693
=============================== WEIGHT =====================================================
    Obs   HUBER_T   RamsayE    Hampel  AndrewWave    Trukey
0     1  0.573964  0.373949  0.776957    0.748954  0.698281
1     2  1.000000  0.929335  1.000000    0.998921  0.998783
2     3  1.000000  0.871744  1.000000    0.996159  0.995072
3     4  0.663549  0.429517  0.872002    0.797457  0.757121
4     5  1.000000  0.826342  1.000000    0.977115  0.973145
5     6  1.000000  0.931786  1.000000    0.999214  0.999071
6     7  1.000000  0.946584  1.000000    0.994242  0.993039
7     8  1.000000  0.825335  1.000000    0.990282  0.988038
8 